# Stage 6 — Circuit Analysis

Two experiments using only the frozen CTLS checkpoint — no new training.

**Why not activation maximization?**
The backbone hook computes `tensor.mean(dim=[2,3])` (global average pool) before storing
each trajectory step (`backbone.py:95`). This makes `d(traj)/d(x[h,w])` identical for
every spatial position, so gradient-based pixel optimisation always converges to uniform
gray — this cannot be fixed by regularisation or initialisation.

**Experiment 1 — Circuit Nearest Neighbors:**
For any target in circuit space (class centroid, interpolation point, confusion-zone point),
retrieve the real validation images with the closest circuit embeddings. No synthesis
required — the model selects its own most-representative examples.

**Experiment 2 — Circuit vs Output Neighbors + GradCAM:**
For boundary images, compare 5-nearest neighbours in circuit space vs output space.
GradCAM on the last spatial feature map (before pooling) reveals which spatial regions
drive the circuit similarity. The per-channel gradient weights `alpha_c` differ across
channels, so the weighted spatial map is non-uniform even though the trajectory is pooled.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from evaluation.circuit_analysis import CircuitAnalyzer, denormalize, CIFAR10_CLASSES
from data.cifar import get_standard_loaders

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

with open('configs/ctls.yaml') as f:
    config = yaml.safe_load(f)

mcfg = config['model']
ecfg = mcfg['meta_encoder']

# Use the final annealed temperature (0.1), not the init value in ctls.yaml.
soft_mask = SoftMask(init_temperature=0.1)
backbone = CTLSBackbone(
    arch=mcfg['arch'], num_classes=mcfg['num_classes'],
    soft_mask=soft_mask, pretrained=False,
).to(DEVICE)
meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    hidden_dim=ecfg.get('hidden_dim', 256),
    embedding_dim=ecfg.get('embedding_dim', 64),
    encoder_type=ecfg.get('encoder_type', 'mlp'),
).to(DEVICE)

ckpt = torch.load('experiments/ctls/best.pt', map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state'])
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
backbone.eval()
meta_encoder.eval()
for p in backbone.parameters():     p.requires_grad_(False)
for p in meta_encoder.parameters(): p.requires_grad_(False)
print(f"Loaded: epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=128,
    num_workers=dcfg.get('num_workers', 4), augment=False, download=True,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, DEVICE)

print("Collecting circuit embeddings...")
z_all, logits_all, x_all, labels_all = analyzer.collect_all(max_samples=10000)
centroids = analyzer.class_centroids(z_all, labels_all)

nearest_cls = torch.stack([z_all @ centroids[c] for c in range(10)], dim=1).argmax(1)
print(f"  {len(z_all)} samples | embedding dim: {z_all.shape[1]}")
print(f"  Nearest-centroid accuracy: {(nearest_cls == labels_all).float().mean():.4f}")

os.makedirs('experiments/stage6', exist_ok=True)

---

## 1. Circuit Nearest Neighbors

For any target point in circuit space, retrieve the real images with the most similar
circuit embeddings (cosine similarity on z). The model selects its own most-representative
examples — no image synthesis required.

Three target types:
- **Class centroid** — 5 real images closest to each class's mean circuit embedding
- **Interpolation path** — nearest real image at each step between two class centroids
- **Confusion zone** — class-A images whose circuit embedding most resembles class-B's centroid

### 1.1 Class Centroid Neighbors

Each row = one class; columns = the 5 real images closest to that class centroid in
circuit space, with cosine similarity shown below. **Red titles** = the retrieved image
belongs to a different class than the centroid — revealing where circuit-space clusters overlap.

In [ ]:
K_CENTROID = 5

fig, axes = plt.subplots(10, K_CENTROID + 1, figsize=((K_CENTROID + 1) * 1.6, 10 * 1.6))
fig.suptitle('Circuit Nearest Neighbors: Class Centroids', fontsize=13, y=1.01)

for cls in range(10):
    _, imgs, lbls, sims = analyzer.nearest_to_target(
        centroids[cls], z_all, x_all, labels_all, k=K_CENTROID
    )
    axes[cls, 0].text(
        0.5, 0.5, CIFAR10_CLASSES[cls],
        ha='center', va='center', fontsize=10, fontweight='bold',
        transform=axes[cls, 0].transAxes,
    )
    axes[cls, 0].axis('off')
    for j in range(K_CENTROID):
        ax  = axes[cls, j + 1]
        lbl = lbls[j].item()
        ax.imshow(denormalize(imgs[j]).permute(1, 2, 0).numpy(), interpolation='nearest')
        ax.set_title(
            f"{CIFAR10_CLASSES[lbl]}\n{sims[j].item():.3f}",
            fontsize=7,
            color='black' if lbl == cls else 'red',
        )
        ax.axis('off')

fig.tight_layout()
fig.savefig('experiments/stage6/centroid_neighbors.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 Interpolation Path Neighbors

Linearly interpolate between two class centroids (L2-normalised at each step) and retrieve
the single nearest real image at each step. The step where the retrieved image's class label
flips is the circuit-space decision boundary for that pair.

In [ ]:
pairs = [
    (5, 3, 'dog_cat',          'dog',        'cat'),
    (1, 9, 'automobile_truck', 'automobile', 'truck'),
    (8, 0, 'ship_airplane',    'ship',       'airplane'),
]
N_INTERP = 8

for cls_a, cls_b, fname, name_a, name_b in pairs:
    alphas = torch.linspace(0, 1, N_INTERP)

    fig, axes = plt.subplots(1, N_INTERP, figsize=(N_INTERP * 1.8, 2.6))
    fig.suptitle(
        f'Interpolation: {name_a} \u2192 {name_b}  (nearest real image per step)',
        fontsize=11,
    )
    for i, alpha in enumerate(alphas):
        a      = alpha.item()
        target = F.normalize(
            (1 - a) * centroids[cls_a] + a * centroids[cls_b], dim=-1
        )
        _, imgs, lbls, sims = analyzer.nearest_to_target(
            target, z_all, x_all, labels_all, k=1
        )
        step_lbl = name_a if i == 0 else (name_b if i == N_INTERP - 1 else f'\u03b1={a:.2f}')
        axes[i].imshow(denormalize(imgs[0]).permute(1, 2, 0).numpy(), interpolation='nearest')
        axes[i].axis('off')
        axes[i].set_title(
            f"{step_lbl}\n{CIFAR10_CLASSES[lbls[0].item()]} {sims[0].item():.3f}",
            fontsize=7,
        )
    fig.tight_layout()
    fig.savefig(f'experiments/stage6/interp_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

### 1.3 Confusion Zone Neighbors

For each pair (A, B), find real images from class A whose circuit embedding is closest
to class B's centroid. These are images where the model's internal reasoning most
resembles its reasoning for class B — the most circuit-ambiguous examples.

These images are saved and reused in the GradCAM section (2.3) to show *which spatial
regions* are driving the circuit-level confusion.

In [ ]:
confusion_pairs = [
    (3, 5, 'cat_dog',          'cat',        'dog'),
    (1, 9, 'automobile_truck', 'automobile', 'truck'),
    (4, 7, 'deer_horse',       'deer',       'horse'),
]
N_CONFUSION = 8

# Stored for reuse in the GradCAM section
confusion_data = {}

for cls_a, cls_b, fname, name_a, name_b in confusion_pairs:
    mask_a  = labels_all == cls_a
    z_a     = z_all[mask_a]
    x_a     = x_all[mask_a]

    dists   = 1 - (z_a @ centroids[cls_b].cpu())
    closest = dists.argsort()[:N_CONFUSION]
    imgs    = x_a[closest]
    sims    = 1 - dists[closest]

    confusion_data[fname] = dict(imgs=imgs, cls_b=cls_b, name_a=name_a, name_b=name_b)

    fig, axes = plt.subplots(1, N_CONFUSION, figsize=(N_CONFUSION * 1.7, 2.5))
    fig.suptitle(
        f'Confusion Zone: {name_a} images most circuit-similar to {name_b}',
        fontsize=11,
    )
    for j in range(N_CONFUSION):
        axes[j].imshow(denormalize(imgs[j]).permute(1, 2, 0).numpy(), interpolation='nearest')
        axes[j].set_title(f'sim={sims[j].item():.3f}', fontsize=8)
        axes[j].axis('off')
    fig.tight_layout()
    fig.savefig(f'experiments/stage6/confusion_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

---

## 2. Circuit vs Output Neighbors + GradCAM

**2.1–2.2:** For boundary images, compare 5-nearest neighbours in circuit space vs output
space and flag cases where the two spaces disagree on the dominant class.

**2.3:** GradCAM on the confusion-zone images from Section 1.3. For each confused image,
compute the gradient of `cos_sim(z, centroid_B)` w.r.t. the last spatial feature map
(before pooling). The per-channel gradient weights `alpha_c` differ because each channel
contributes differently to the pooled trajectory → z → cosine similarity. Applying
these weights to the full spatial map reveals which regions are most circuit-relevant
for the rival class.

In [ ]:
K = 5

# Select 2 most circuit-confused images per class (20 total)
query_indices = []
for cls in range(10):
    mask       = labels_all == cls
    global_idx = torch.where(mask)[0]
    z_cls      = z_all[mask]

    own_dist   = 1 - (z_cls @ centroids[cls].cpu())
    other_dist = torch.stack(
        [1 - (z_cls @ centroids[c].cpu()) for c in range(10) if c != cls], dim=1
    ).min(dim=1).values

    top2 = (own_dist - other_dist).argsort(descending=True)[:2]
    query_indices.extend(global_idx[top2].tolist())

query_indices        = torch.tensor(query_indices)
all_centroid_sims    = torch.stack([z_all @ centroids[c].cpu() for c in range(10)], dim=1)
nearest_centroid_cls = all_centroid_sims.argmax(dim=1)
print(f"Selected {len(query_indices)} query images (2 per class)")

### 2.1 KNN Comparison Grid

In [ ]:
n_q   = len(query_indices)
n_col = 1 + K + K
fig, axes = plt.subplots(n_q, n_col, figsize=(n_col * 1.3, n_q * 1.3))

for row, qi in enumerate(query_indices.tolist()):
    q_z      = z_all[qi]
    q_logits = logits_all[qi]
    q_label  = labels_all[qi].item()

    _, c_imgs, _ = analyzer.knn_circuit(q_z, z_all, x_all, k=K)
    _, o_imgs, _ = analyzer.knn_output(q_logits, logits_all, x_all, k=K)

    ax = axes[row, 0]
    ax.imshow(denormalize(x_all[qi]).permute(1, 2, 0).numpy(), interpolation='nearest')
    ax.axis('off')
    ax.set_ylabel(CIFAR10_CLASSES[q_label], fontsize=7, rotation=0, labelpad=32, va='center')
    if row == 0:
        ax.set_title('Query', fontsize=8, fontweight='bold')

    for j in range(K):
        ax = axes[row, 1 + j]
        ax.imshow(denormalize(c_imgs[j]).permute(1, 2, 0).numpy(), interpolation='nearest')
        ax.axis('off')
        if row == 0: ax.set_title(f'Circ-{j+1}', fontsize=7)

    for j in range(K):
        ax = axes[row, 1 + K + j]
        ax.imshow(denormalize(o_imgs[j]).permute(1, 2, 0).numpy(), interpolation='nearest')
        ax.axis('off')
        if row == 0: ax.set_title(f'Out-{j+1}', fontsize=7)

fig.subplots_adjust(wspace=0.04, hspace=0.04)
fig.suptitle('Circuit Neighbors (Circ-1…5)  vs  Output Neighbors (Out-1…5)',
             fontsize=11, y=1.005)
fig.savefig('experiments/stage6/knn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Divergence Summary

Compares the **dominant class** (mode of 5 neighbours) in circuit vs output space.
DIVERGE = the two spaces disagree on the dominant class.

In [ ]:
def _dominant(nn_idx):
    return torch.bincount(labels_all[nn_idx], minlength=10).argmax().item()

print(f"{'Idx':<6} {'True':<13} {'Near centroid':<14} "
      f"{'Circuit-NN classes':<28} {'Output-NN classes':<28} {'Dom C':<7} {'Dom O':<7} Status")
print('-' * 108)

for qi in query_indices.tolist():
    q_label  = labels_all[qi].item()
    c_idx, _, _ = analyzer.knn_circuit(z_all[qi], z_all, x_all, k=K)
    o_idx, _, _ = analyzer.knn_output(logits_all[qi], logits_all, x_all, k=K)

    c_dom  = _dominant(c_idx)
    o_dom  = _dominant(o_idx)
    near_c = nearest_centroid_cls[qi].item()
    c_str  = ' '.join(CIFAR10_CLASSES[c][:4] for c in sorted(set(labels_all[c_idx].tolist())))
    o_str  = ' '.join(CIFAR10_CLASSES[c][:4] for c in sorted(set(labels_all[o_idx].tolist())))

    print(f"{qi:<6} {CIFAR10_CLASSES[q_label]:<13} {CIFAR10_CLASSES[near_c]:<14} "
          f"{c_str:<28} {o_str:<28} "
          f"{CIFAR10_CLASSES[c_dom][:4]:<7} {CIFAR10_CLASSES[o_dom][:4]:<7} "
          f"{'DIVERGE' if c_dom != o_dom else 'agree'}")

### 2.3 GradCAM on Confusion-Zone Images

For each confused image from Section 1.3, compute GradCAM of `cos_sim(z, centroid_B)`
w.r.t. the last ResNet block's spatial feature map.

**Top row:** original image. **Bottom row:** GradCAM overlay (jet colormap, hot = most
circuit-relevant for the rival class).

**GradCAM on the animal body** = genuine feature-based confusion (shared shape/texture).  
**GradCAM on the background** = spurious confusion driven by scene context, not the object.

In [ ]:
def overlay_cam(img_01, cam):
    """Blend [3,H,W]∈[0,1] with jet heatmap [H,W]∈[0,1]. Returns [H,W,3]."""
    return np.clip(
        0.55 * img_01.permute(1, 2, 0).numpy() + 0.45 * cm.jet(cam.numpy())[:, :, :3],
        0, 1,
    )

print("Computing GradCAM maps...")

for fname, entry in confusion_data.items():
    imgs   = entry['imgs']
    cls_b  = entry['cls_b']
    name_a = entry['name_a']
    name_b = entry['name_b']
    n      = len(imgs)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.7, 4.0))
    fig.suptitle(
        f'GradCAM: which {name_a} pixels activate {name_b}-like circuits',
        fontsize=11,
    )
    for j in range(n):
        cam = analyzer.gradcam(imgs[j], centroids[cls_b])
        axes[0, j].imshow(denormalize(imgs[j]).permute(1, 2, 0).numpy(), interpolation='nearest')
        axes[0, j].axis('off')
        axes[1, j].imshow(overlay_cam(denormalize(imgs[j]), cam))
        axes[1, j].axis('off')
        print(f"  {fname} {j+1}/{n}", end='\r')
    print()

    axes[0, 0].set_ylabel('Original', fontsize=9)
    axes[1, 0].set_ylabel('GradCAM', fontsize=9)
    fig.tight_layout()
    fig.savefig(f'experiments/stage6/gradcam_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

---

## Summary

| Section | What it shows |
|---|---|
| 1.1 Centroid neighbors | Which real images the model considers most class-typical in circuit space. Red titles reveal semantic overlap between class clusters. |
| 1.2 Interpolation path | Where the circuit-space decision boundary lies. The step where the retrieved label flips is the boundary between the two classes. |
| 1.3 Confusion zone | Real class-A images that are closest to class-B's centroid internally — the most ambiguous examples. |
| 2.1 KNN comparison | Side-by-side circuit vs output neighbours for boundary images. |
| 2.2 Divergence table | Dominant-class comparison between the two neighbour sets. DIVERGE = internal representation and output prediction are misaligned. |
| 2.3 GradCAM | Spatial heatmaps showing which pixels drive circuit similarity to the rival class. Distinguishes genuine shape-based confusion from spurious background/texture confusion. |

**Next directions:**
- Idea 3 (Robustness Auditing): apply input noise, recompute z, use centroid-neighbors
  of the perturbed embedding to show what the model 'thinks it sees' under distribution shift.
- Idea 4 (Divergence Analysis): find the specific trajectory layer where the circuit
  diverges from the expected class path and run GradCAM at that layer.